In [ ]:


import numpy as np
import pandas as pd






import ast
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
from config import *
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in Path(folder_loc).parents if (p / 'jar_locations.py').is_file())))
from generation import generate_arguments_string, run_simulation
pop_sizes_loc = next(p for p in Path(folder_loc).parents if (p / 'jar_locations.py').is_file()) / 'pop_sizes' / 'degree_distribution'

from jar_locations import simEMoran_jar


In [ ]:
def fix_list(pieces):
    s = " ".join(pieces)
    return ast.literal_eval(s)

def generate_trajectory(graph,popsize,d,r=1,generate = True,timestep=0.001,repeat_num=0,regular = False):
    relax = int(timestop/10)
    sim_ended_early=True
    counter = 0
    temp_timestep = timestep
    while sim_ended_early and counter<20:
        output_name = f"CC-N-{popsize}-geom-{graph}-d-{d}-r-{r}-tr-{1}-repeat-{repeat_num}"
        if generate:
            run_simulation(results_loc, output_name, simEMoran_jar, generate_arguments_string({
                'module': 'eMoran',
                'delay': 0,
                'timestep': temp_timestep,
                'popsize': popsize,
                'popupdate': 'e',
                'mutation': 0,
                'geometry': graph,
                'fitness': f'{r},{r}',
                'deathrate': d,
                'init': ('mono', '0,1'),
                'timestop': timestop,
                'timerelax': 0,
            }))
        with open( f'{results_loc}{output_name}.txt', 'r') as file:
            lines = file.readlines()
        na_data_lines = []
        degree_dist_lines = []
        all_degree_dist = None
        for line in lines:
            line = line.strip()
            if line[:len("Na")]=="Na" and line[:len("Nav")]!="Nav":
                na_data_lines.append(line.split(' ')[1].replace('\t',''))
            elif line[:len("Inhabited Degree Distribution")]=="Inhabited Degree Distribution":
                degree_dist_lines.append(line.split(' ')[3:])
            elif line[:len("All Degrees")]=="All Degrees":
                all_degree_dist = fix_list(line.split(' ')[2:])
        df = pd.DataFrame({'na': na_data_lines,
                            'deg': degree_dist_lines,
                                })
        if len(df)>10:
            return df.iloc[relax:],all_degree_dist  
        else:
            print("Failed early")
            counter+=1
    return df, all_degree_dist


def remove_empty_bins(bins, data):
    data = np.asarray(data)
    bins = list(bins)

    changed = True
    while changed and len(bins) > 2:
        changed = False
        counts, edges = np.histogram(data, bins=bins)
        for i, c in enumerate(counts):
            if c == 0:
                if i < len(counts) - 1:
                    del bins[i + 1]
                else:
                    
                    del bins[i]
                changed = True
                break 
    return np.array(bins)

def loglog_fit(bin_edges, counts, fit_range=None):
    bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])
    mask = counts > 0
    if fit_range is not None:
        mask &= (bin_centers >= fit_range[0]) & (bin_centers <= fit_range[1])

    log_x = np.log(bin_centers[mask])
    log_y = np.log(counts[mask])
    slope, intercept = np.polyfit(log_x, log_y, 1)

    fit_x = bin_centers[mask]
    fit_y = np.exp(intercept) * fit_x**slope
    return slope, intercept, fit_x, fit_y

In [ ]:
ks = [3,4,6,8]
generate = False
adj_degs = []
f_or_r = "f"
ds = [0.2,0.5]
dfs = {(k,d):[] for k in ks for d in ds}
degree_dists = {(k,d):[] for k in ks for d in ds}
repeat_num=0
num_repeats = 10
for k in ks:
    for d in ds:
        print(k,d)
        with open(pop_sizes_loc / f"pop_size_{f_or_r}{k}_d_{d}.txt", 'r') as f:
            N = int(f.read())
        graph = f"{f_or_r}{k}"
        original_timestep =1
        for repeat_num in range(num_repeats):
            df, all_degree_dist = generate_trajectory(graph,N,d,generate=generate,timestep=1,repeat_num=repeat_num)
            if df is None:
                continue
            df['na']=df['na'].astype(float)
            df['deg']=df['deg'].apply(fix_list)
            dfs[(k,d)].append(df)
            degree_dists[(k,d)].append(all_degree_dist)


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8, 10), sharex=True, sharey=True)
axes_flat = axes.flatten()
plot_idx = 0

obs_slopes = {d: [] for d in ds}
graph_slopes = {d: [] for d in ds}

COLORS =['#3b8fd9', '#e8618c']
resident_color = COLORS[0]
graph_color = COLORS[1]


ks = [3,4,8]
for row_idx, k in enumerate(ks):
    for col_idx, d in enumerate(ds):
        ax = axes[row_idx, col_idx]
        print(k, d)
        with open(pop_sizes_loc / f"pop_size_{f_or_r}{k}_d_{d}.txt", 'r') as f:
            M = int(f.read())
        all_graph_degs = []
        all_observed_degs_counts = {}
        for df, degree_dist in zip(dfs[(k, d)], degree_dists[(k, d)]):
            for deg_dist, na in zip(df['deg'], df['na']):
                for num, val in enumerate(deg_dist):
                    if num not in all_observed_degs_counts:
                        all_observed_degs_counts[num] = 0
                    all_observed_degs_counts[num] += val * na

            for degree_dist in degree_dists[(k, d)]:
                for num, val in enumerate(degree_dist):
                    for _ in range(int(val)):
                        all_graph_degs.append(num)

        cc = df['na'].mean() / sum(degree_dist)
        print(all_observed_degs_counts)
        plot_idx += 1

        min_degree = min(all_graph_degs)
        max_degree = max(all_graph_degs)
        num_bins = 15
        log_bins = np.logspace(np.log10(min_degree), np.log10(max_degree), num=num_bins)
        log_bins = np.unique(np.round(log_bins).astype(int))
        bins = remove_empty_bins(log_bins, all_graph_degs)

        val, weights = zip(*[(kk, v) for kk, v in all_observed_degs_counts.items()])

        obs_counts, obs_edges, _ = ax.hist(val, weights=weights, bins=bins, density=True,
                                    alpha=0.5,linewidth=1.8,
                                    color=resident_color, label="Resident distribution")
        graph_counts_h, graph_edges, _ = ax.hist(all_graph_degs, bins=bins, density=True,
                                    alpha=0.5,linewidth=1.8, linestyle='--',
                                    color=graph_color, label="Graph distribution")

        obs_slope, obs_intercept, obs_fx, obs_fy = loglog_fit(obs_edges, obs_counts)
        graph_slope, graph_intercept, graph_fx, graph_fy = loglog_fit(graph_edges, graph_counts_h)

        obs_slopes[d].append(obs_slope)
        graph_slopes[d].append(graph_slope)

        ax.plot(obs_fx, obs_fy, '--', color='black', linewidth=1.5,
                label=f"Resident fit")
        ax.plot(graph_fx, graph_fy, linestyle=(0, (1, 1)), color='black', linewidth=2,
                label=f"Graph fit")

        ax.text(0.05, 0.05,
        f"Resident exponent: {obs_slope:.2f}\nGraph exponent: {graph_slope:.2f}",
        transform=ax.transAxes, fontsize=8, va='bottom', ha='left',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none'))

        ax.set_yscale('log')
        ax.set_xscale('log')
        ax.set_xlabel('Degree')
        ax.set_ylabel('Density')

for col_idx, d in enumerate(ds):
    axes[0, col_idx].set_title(f"$d={d}$")

for row_idx, k in enumerate(ks):
    axes[row_idx, 0].annotate(
        fr'$\langle k \rangle = {k}$',
        xy=(0, 0.5), xycoords='axes fraction',
        xytext=(-axes[row_idx, 0].yaxis.labelpad - 40, 0),
        textcoords='offset points',
        ha='right', va='center', rotation=90, fontsize=11
    )
handles, labels = axes_flat[plot_idx - 1].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=4)
fig.tight_layout()

plt.savefig('scale_free_occupancy_by_degree_grid.pdf', bbox_inches='tight')